## Setup runner & utilities

In [8]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

run_start_server: run_start_server | None = None

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/nanotube.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="path tools test")
if run_start_server:
    print("starting server")
    imd_runner.load(0)

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities
utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [ ]:
TRAIL_COLOR = [0.0, 1.0, 1.0, 0.5]
PATH_COLOR = [1.0, 1.0, 0.0, 0.5]
HOVER_COLOR = [1.0, 0.0, 0.0, 0.5]
CHOSEN_COLOR = [0.0, 1.0, 0.0, 0.5]

PATH_WIDTH = 0.04
TRAIL_WIDTH = 0.02
HOVER_WIDTH = 0.06
CHOSEN_WIDTH = 0.07

SELECTION_RADIUS = 0.2
TRAIL_SIMPLIFICATION_FACTOR = 0.05
PATH_SIMPLIFICATION_FACTOR = 0.05

In [ ]:
from follower import PathFollowerAgent, Path
import numpy as np


PATHS: dict[str, Path] = {}
NEXT_PATH_ID = 0
FOLLOWER_AGENT: PathFollowerAgent | None = None


def CREATE_PATH():
    global NEXT_PATH_ID
    path_id = str(NEXT_PATH_ID)
    NEXT_PATH_ID += 1
    PATHS[path_id] = Path.empty()
    return path_id


def REDRAW_PATH(path_id: str):
    path = PATHS[path_id]
    PATH_OBJECTS.update_line(f"path.{path_id}", type="dashed", positions=path.positions, sgtize=PATH_WIDTH, color=PATH_COLOR)


def STOP_FOLLOWER():
    global FOLLOWER_AGENT
    if FOLLOWER_AGENT is not None:
        FOLLOWER_AGENT.close()
    FOLLOWER_AGENT = None


def START_FOLLOWER(particles: list[int], path: Path):
    global FOLLOWER_AGENT
    STOP_FOLLOWER()
    FOLLOWER_AGENT = PathFollowerAgent.from_runner(imd_runner)
    FOLLOWER_AGENT.set_path(path)
    FOLLOWER_AGENT.set_particles(particles)
    FOLLOWER_AGENT.start()


def SIMPLIFY_LINE(points: list[list[float]], min_distance: float = 0.05):
    if len(points) < 2:
        return points

    simplified = [points[0]]
    last_point = np.array(points[0])

    for point in points[1:]:
        current_point = np.array(point)
        distance = np.linalg.norm(current_point - last_point)

        if distance >= min_distance:
            simplified.append(point)
            last_point = current_point

    return simplified


def intersect_trails(position, radius):
    for path_id, path in TRAIL_POINTS.items():
        if intersects_sphere_line(position, radius, np.array(path)):
            return path_id, path
    return None


def intersect_paths(position, radius):
    for path_id, path in PATHS.items():
        if intersects_sphere_line(position, radius, path.positions):
            return path_id, path
    return None


def get_interaction_centroid(interaction: ParticleInteraction):
    indexes = [int(index) for index in interaction.particles]
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    centroid = [float(x) for x in np.mean(positions[indexes], axis=0)]
    return centroid


def segment_length(points):
    pts = np.asarray(points, dtype=float)
    if len(pts) < 2:
        return []
    return [float(np.linalg.norm(pts[i + 1] - pts[i])) for i in range(len(pts) - 1)]

def line_length(points):
    return float(sum(segment_length(points)))



# Draw path mode

In [ ]:
from nanover.jupyter import Mode, SceneObjectsUtility
from follower import Path

PATH_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)
CURSOR_DRAWING_PATH_ID = {}


class DrawPathMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            CURSOR_DRAWING_PATH_ID[key] = CREATE_PATH()

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            CURSOR_DRAWING_PATH_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        if not key in CURSOR_DRAWING_PATH_ID:
            return

        path_id = CURSOR_DRAWING_PATH_ID[key]

        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        PATHS[path_id].append_position(next_pos)

        REDRAW_PATH(path_id)


utilities.use_interaction_modes()
utilities.add_interaction_mode(DrawPathMode, "draw path", icon="✏️")

# Trail mode
- draws a line at COMs when interacting with residue
- [ ] missing instructions in UI
- [ ] missing select-delete trail (+ button in UI)
- simplification at the end

workflow (missing):
- enter trail mode
- manipulate > that creates trails
- mode UI panel: [delete trail] [delete all]
    - [delete trail] selected a trail line, press A to delete
    - [transform to path]

In [ ]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode

TRAIL_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)
TRAIL_POINTS = {}


class InteractionTrailMode(Mode):
    utilities.notify_all(f"[trigger]: draw interaction trail\n[A/X]: simplify last trail\n[B/Y]: remove last trail")
    interacting = False

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        TRAIL_POINTS[key] = []
        self.interacting = True

    def on_interaction_updated(self, *, key: str, interaction: ParticleInteraction):
        TRAIL_POINTS[key].append(get_interaction_centroid(interaction))
        TRAIL_OBJECTS.update_line(f"trail.{key}", positions=TRAIL_POINTS[key], size=TRAIL_WIDTH, color=TRAIL_COLOR)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        self.interacting = False
        # before = len(TRAIL_POINTS[key]) # from 360 to 45 points for 0.05 simplification factor
        # before = line_lenght(TRAIL_POINTS[key]) # from 6.65.... to 6.59....
        TRAIL_POINTS[key] = SIMPLIFY_LINE(TRAIL_POINTS[key], min_distance=TRAIL_SIMPLIFICATION_FACTOR)
        traj_len = line_length(TRAIL_POINTS[key])
        TRAIL_OBJECTS.update_line(f"trail.{key}", positions=TRAIL_POINTS[key], size=TRAIL_WIDTH, color=TRAIL_COLOR, lenght=traj_len)
        utilities.objects.update_label(f"{key}", text=f"{traj_len:.4}nm", position=tuple(TRAIL_POINTS[key][-1]))

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if self.interacting or button not in ["primary", "secondary"]:
            return
        
        cursor_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_trails(cursor_pos, SELECTION_RADIUS)

        if hovered is None:
            return

        path_id, path = hovered

        if button == "primary":
            new_path_id = CREATE_PATH()
            print(f"{new_path_id}")
            path = PATHS[new_path_id]
            for point in TRAIL_POINTS[path_id]:
                path.append_position(point)
            REDRAW_PATH(new_path_id)

            TRAIL_POINTS.pop(path_id, None)
            TRAIL_OBJECTS.remove_line(f"trail.{path_id}") # remove  trail

            utilities.notify_all(f"The trail is now a path")

        if button == "secondary":
            TRAIL_POINTS.pop(path_id, None)
            TRAIL_OBJECTS.remove_line(f"trail.{path_id}")
            utilities.objects.remove_label(f"{path_id}") ## NEED TO TEST THIS
            utilities.notify_all(f"Trail deleted")
    
    def on_cursor_updated(self, *, key: str, cursor: dict):
        if self.interacting:
            return

        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_trails(next_pos, SELECTION_RADIUS)

        if hovered is None:
            SELECTION_OBJECTS.remove_line(f"hover.{key}")
        else:
            SELECTION_OBJECTS.update_line(f"hover.{key}", positions=hovered[1], size=HOVER_WIDTH, color=HOVER_COLOR)
            utilities.notify_all(f"[A] convert to path\n\n[B] to delete trail")


utilities.add_interaction_mode(InteractionTrailMode, "trails", icon="〰️")

In [ ]:
TRAIL_POINTS.items()
# TRAIL_POINTS['interaction.8acb13ca-98f0-4dc7-9186-e450a4677f13'][-1]

# len(TRAIL_POINTS['interaction.f4f11f3e-930a-4536-8fca-a5d564ef0093'])

# TRAIL_OBJECTS['interaction.ac04d265-1b44-454d-bf30-783af8c1006a']


# TRAIL_POINTS['interaction.ac04d265-1b44-454d-bf30-783af8c1006a']

#  [1.818497657775879, 2.3373303413391113, 2.544994592666626]]

# get last point
# TRAIL_POINTS['interaction.ac04d265-1b44-454d-bf30-783af8c1006a'][-1]

# utilities.objects._buffer
# utilities.objects.update_label("a12",text="aaa")



# Path following mode
- start path follower
- flow
    - [ ] missing instructions on UI
    - [ ] missing select force type on UI
    - [ ] missing set force scale on UI
    - [ ] missing run / pause button on UI
    - [ ] missing stop button on UI
    - enter path selection model (highlight near path)
    - select residue on click
    - start path follow process [[follower.py]]
    - [ ] missing ending callback

In [ ]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from intersection import intersects_sphere_line


SELECTION_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)

class PathFollowingMode(Mode):
    PATH_ID = None
    PARTICLES = None

    def check(self):
        if self.PARTICLES is None or self.PATH_ID is None:
            return

        utilities.notify_all(f"FOLLOWING!")
        START_FOLLOWER(self.PARTICLES, PATHS[self.PATH_ID])
        self.PATH_ID = None
        self.PARTICLES = None

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        self.PARTICLES = [int(index) for index in interaction.particles]
        utilities.notify_all(f"SELECTED PARTICLES: {self.PARTICLES}")
        self.check() 

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button != "primary":
            return

        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_paths(next_pos, SELECTION_RADIUS)

        if hovered is not None:
            path_id, path = hovered
            utilities.notify_all(f"SELECTED PATH {path_id}")
            self.PATH_ID = path_id
            SELECTION_OBJECTS.update_line(f"selected.path", positions=path.positions, size=CHOSEN_WIDTH, color=CHOSEN_COLOR)
            self.check()

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_paths(next_pos, SELECTION_RADIUS)

        if hovered is None:
            SELECTION_OBJECTS.remove_line(f"hover.{key}")
        else:
            SELECTION_OBJECTS.update_line(f"hover.{key}", positions=hovered[1].positions, size=HOVER_WIDTH, color=HOVER_COLOR)

utilities.add_interaction_mode(PathFollowingMode, "follow", icon="🚂")

# Trail2Path mode
- convert trail into a path (select trail, copy as trail, delete original)
- [ ] missing instructions in UI
- [ ] missing undo button on UI

In [ ]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from traceback import print_exc


class Trail2PathMode(Mode):

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button != "primary":
            return

        cursor_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_trails(cursor_pos, SELECTION_RADIUS)

        if hovered is not None:
            path_id, path = hovered

            new_path_id = CREATE_PATH()
            print(f"{new_path_id}")
            path = PATHS[new_path_id]
            for point in TRAIL_POINTS[path_id]:
                path.append_position(point)
            REDRAW_PATH(new_path_id)

            TRAIL_POINTS.pop(path_id, None)
            TRAIL_OBJECTS.remove_line(f"trail.{path_id}") # remove  trail

            utilities.notify_all(f"Trail transformed")

    def on_cursor_updated(self, *, key: str, cursor: dict):
        cursor_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_trails(cursor_pos, SELECTION_RADIUS)

        if hovered is None:
            SELECTION_OBJECTS.remove_line(f"hover.{key}")
        else:
            SELECTION_OBJECTS.update_line(f"hover.{key}", positions=hovered[1], size=HOVER_WIDTH, color=HOVER_COLOR)

utilities.add_interaction_mode(Trail2PathMode, "trail2path", icon="🛤️")

In [ ]:
utilities.show_logging()

# Identification
not part of the experiments, but useful for developing them


In [ ]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from nanover.mdanalysis import frame_data_to_mdanalysis

# prepare info from simulation
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

class InfoMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        atom = universe.atoms[interaction.particles[0]]
        utilities.notify_all(str(atom))

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass

utilities.add_interaction_mode(InfoMode, "info", icon="ℹ️")

# Display grid
command, not mode
- toggles visibility of grids around you
- [ ] specify grid size on UI

# Experiment draft

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities
from nanover.imd import ParticleInteraction
from nanover.jupyter import ImdAgent
from nanover.trajectory import FrameData


utilities = NanoverJupyterUtilities.from_runner(imd_runner)

utilities.objects.update_shape("target", position=(3.0, 3.0, 3.0), color=[0.0, 1.0, 0.0, 0.2], size=1.0)


In [ ]:
import numpy as np

def particle_inside_sphere(particle_pos: np.ndarray, sphere_pos: np.ndarray, sphere_radius: float) -> bool:
    delta = np.asarray(particle_pos, dtype=float) - np.asarray(sphere_pos, dtype=float)
    return float(np.dot(delta, delta)) <= sphere_radius * sphere_radius

'''
testing
'''
test_pos = [2.7, 2.8, 3.0]
test_zon = [3.0, 3.0, 3.0]
test_rad = 0.5
print(particle_inside_sphere(test_pos, test_zon, test_rad))
print(particle_inside_sphere(test_pos, test_zon, test_rad / 2))

In [ ]:


class ZoneEvaluator(ImdAgent):
    def set_goal(self, goal: np.ndarray, radius: float):
        self.goal = goal
        self.radius = radius

    def set_particles(self, particles: list[np.ndarray]):
        self.particles = particles

    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        for atom in self.particles:
            pos = full_frame.particle_positions[atom]
            if particle_inside_sphere(pos, self.goal, self.radius):
                utilities.objects.update_shape("target", position=(3.0, 3.0, 3.0), color=[0.4, 0.3, 0.9, 0.3], size=1.0)
                break
            else:
                utilities.objects.update_shape("target", position=(3.0, 3.0, 3.0), color=[0.0, 1.0, 0.0, 0.2], size=1.0)
                break


In [ ]:
import numpy as np

ZONE_EVALUATOR: ZoneEvaluator | None = None

def STOP_EXP_TARGET():
    global ZONE_EVALUATOR
    if ZONE_EVALUATOR is not None:
        ZONE_EVALUATOR.close()
    ZONE_EVALUATOR = None


def START_EXP_TARGET(particles: list[int], target_position: np.ndarray, target_radius: float):
    global ZONE_EVALUATOR
    STOP_FOLLOWER()
    ZONE_EVALUATOR = ZoneEvaluator.from_runner(imd_runner)
    ZONE_EVALUATOR.set_goal(target_position, target_radius)
    ZONE_EVALUATOR.set_particles(particles)
    ZONE_EVALUATOR.start()

def test():
    print("runningx")
    START_EXP_TARGET(list(range(60,64)), (3.0, 3.0, 3.0), 0.5)

imd_runner.app_server.register_command("user/zone eval on", test, icon="🧪🟢")
imd_runner.app_server.register_command("user/zone eval off", STOP_EXP_TARGET, icon="🧪🔴")
